# 01 -- Main Experiment: TP53 Mutation Subtype CNN (21bp)

Trains two independent `SimpleCNN` models (from `src/model.py`) on the
21bp-window datasets produced by `notebooks/data_pipeline.ipynb` (notebook
00): one on hotspot positions (`n_instances >= 100`), one on rare-variant
positions (`n_instances < 100`). Each is compared against four baselines
(random, majority class, CpG rule, trinucleotide logistic regression) and a
theoretical position-level accuracy ceiling, with McNemar's test for
statistical comparison.

**Inputs verified against notebook 00's actual output** (see the schema
verification cell below) -- not assumed:
- `data/splits/window_21/{hotspot,rare}/{train,val,test}.csv`
  (columns: `position_id, Sequence, MutationType`)
- `data/processed/position_table.csv`
  (columns: `position_id, gene_number, cds_pos, n_instances, majority_subtype,
  n_distinct_subtypes, shannon_entropy, hotspot_flag`)

**Schema note:** the position table has no `Sequence` column, so the CpG
flag and the position-level accuracy ceiling are computed directly from each
split's own `Sequence`/`MutationType` columns (see `src/data.py`), not by
joining against the position table. The position table is still loaded here
and cross-checked against the hotspot/rare split assignment for consistency.

**Cluster note:** notebook 00 groups `position_id`s that share identical
sequence at the smallest window size (11bp) into locus clusters, because
TP53's 19 transcript isoforms cause the same genomic variant to be
re-annotated (and re-counted) once per affected transcript. `hotspot_flag`
is therefore a **cluster-level** concept, not a position-level one --
`data/processed/cluster_table.csv` (not `position_table.csv`) is the
source of truth for it, joined via `position_table.cluster_id`.

Reusable logic lives in `src/data.py`, `src/model.py`, `src/train.py`,
`src/metrics.py` -- this notebook mostly wires them together, since
`02_window_ablation.ipynb` imports the same modules for the 11/51/101bp runs.


In [1]:
SEED = 42
import random, numpy as np, torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
import os
import sys
import json

import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data import (
    CLASSES, CLASS_TO_IDX, load_split, load_position_table, load_cluster_table,
    encode_labels, MutationDataset, compute_is_cpg, position_majority_ceiling,
)
from src.model import SimpleCNN, count_trainable_params
from src.train import compute_class_weights, train_model, predict
from src.metrics import (
    compute_all_metrics, plot_confusion_matrix, random_baseline, majority_baseline,
    cpg_baseline, logistic_regression_baseline, mcnemar_test,
    cluster_paired_bootstrap_test, cluster_wilcoxon_paired_test,
)

WINDOW_SIZE = 21
DATASETS = ['hotspot', 'rare']

SPLITS_DIR = os.path.join(PROJECT_ROOT, 'data', 'splits', f'window_{WINDOW_SIZE}')
POSITION_TABLE_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'position_table.csv')
CLUSTER_TABLE_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'cluster_table.csv')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'main')

for name in DATASETS:
    os.makedirs(os.path.join(RESULTS_DIR, name), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Project root:  {PROJECT_ROOT}")
print(f"Splits dir:    {SPLITS_DIR}")
print(f"Position table:{POSITION_TABLE_PATH}")
print(f"Cluster table: {CLUSTER_TABLE_PATH}")
print(f"Results dir:   {RESULTS_DIR}")
print(f"Device:        {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == 'cuda' else ""))
print(f"Classes:       {CLASSES}")


Project root:  C:\Users\danya\Documents\projects\tp53_mutation_subtype
Splits dir:    C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\splits\window_21
Position table:C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\position_table.csv
Cluster table: C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\cluster_table.csv
Results dir:   C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main
Device:        cuda (NVIDIA GeForce RTX 5090)
Classes:       ['C>A', 'C>G', 'C>T', 'T>A', 'T>C', 'T>G']


## Load and verify inputs

`load_split` / `load_position_table` (in `src/data.py`) assert the exact
columns expected from notebook 00, assert there is no leftover `N` padding,
and assert every `MutationType` value is one of the six known classes --
this is where a schema drift in notebook 00 would surface loudly instead of
silently.


In [3]:
splits = {}
for name in DATASETS:
    splits[name] = {
        split: load_split(os.path.join(SPLITS_DIR, name, f'{split}.csv'))
        for split in ('train', 'val', 'test')
    }
    for split, df in splits[name].items():
        print(f"{name:>7}/{split:<5}: {len(df):>7,} rows, "
              f"{df['position_id'].nunique():>5,} positions, seq_len={df['Sequence'].str.len().iloc[0]}")

position_table = load_position_table(POSITION_TABLE_PATH)
cluster_table = load_cluster_table(CLUSTER_TABLE_PATH)
print(f"\nPosition table: {len(position_table):,} rows -> {POSITION_TABLE_PATH}")
print(f"Cluster table:  {len(cluster_table):,} rows -> {CLUSTER_TABLE_PATH}")

# Cross-check: every position_id in the hotspot/rare splits should map
# (position_table.cluster_id -> cluster_table.hotspot_flag) to the expected
# hotspot/rare group, and the split's own position_id set should be disjoint
# between hotspot and rare (as guaranteed by notebook 00, re-verified here
# rather than assumed).
position_to_cluster = position_table.set_index('position_id')['cluster_id']
cluster_hotspot_flag = cluster_table.set_index('cluster_id')['hotspot_flag']

for name in DATASETS:
    all_ids = set()
    for split, df in splits[name].items():
        all_ids |= set(df['position_id'])
    expected_flag = (name == 'hotspot')
    actual_flags = cluster_hotspot_flag.loc[position_to_cluster.loc[list(all_ids)]]
    assert (actual_flags == expected_flag).all(), (
        f"{name}: some position_ids don't map to hotspot_flag={expected_flag} via cluster_table"
    )

hotspot_ids = set()
for split, df in splits['hotspot'].items():
    hotspot_ids |= set(df['position_id'])
rare_ids = set()
for split, df in splits['rare'].items():
    rare_ids |= set(df['position_id'])
assert not (hotspot_ids & rare_ids), "hotspot and rare position_id sets overlap"

# And the leakage fix itself: no exact-duplicate Sequence may cross a
# train/val/test boundary, for either dataset, at this window size.
for name in DATASETS:
    seq_by_split = {split: set(df['Sequence']) for split, df in splits[name].items()}
    assert not (seq_by_split['train'] & seq_by_split['val']), f"{name}: Sequence overlap train/val"
    assert not (seq_by_split['train'] & seq_by_split['test']), f"{name}: Sequence overlap train/test"
    assert not (seq_by_split['val'] & seq_by_split['test']), f"{name}: Sequence overlap val/test"

print("\nOK: hotspot/rare position_ids agree with cluster_table.hotspot_flag, are disjoint, "
      "and no Sequence crosses a train/val/test boundary.")


hotspot/train: 467,431 rows, 5,304 positions, seq_len=21
hotspot/val  :  97,173 rows, 1,143 positions, seq_len=21
hotspot/test : 112,511 rows, 1,136 positions, seq_len=21
   rare/train:  11,274 rows, 4,041 positions, seq_len=21
   rare/val  :   2,677 rows,   915 positions, seq_len=21
   rare/test :   2,362 rows,   884 positions, seq_len=21

Position table: 13,423 rows -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\position_table.csv
Cluster table:  906 rows -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\cluster_table.csv



OK: hotspot/rare position_ids agree with cluster_table.hotspot_flag, are disjoint, and no Sequence crosses a train/val/test boundary.


## Model architecture

`SimpleCNN` (from `src/model.py`) is window-size agnostic (global max pool
over the sequence dimension, no hardcoded `seq_len`), which is what lets
`02_window_ablation.ipynb` reuse this exact class unchanged at 11/51/101bp.


In [4]:
_sanity_model = SimpleCNN()
n_params = count_trainable_params(_sanity_model)
print(_sanity_model)
print(f"\nTrainable parameters: {n_params:,}")
assert n_params == 7206, f"Expected 7,206 trainable parameters, got {n_params:,}"
del _sanity_model


SimpleCNN(
  (conv1): Conv1d(4, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=64, out_features=6, bias=True)
  (relu): ReLU(inplace=True)
)

Trainable parameters: 7,206


## Train: hotspot and rare-variant models, independently, from scratch

Same architecture and hyperparameters for both (Adam lr=1e-4, batch_size=128,
max_epochs=50, cross-entropy with inverse-frequency class weights, early
stopping on validation accuracy with patience=15, ReduceLROnPlateau on
validation loss, gradient clipping max_norm=1.0, FP16 mixed precision on
CUDA).


In [5]:
models = {}
histories = {}

for name in DATASETS:
    print("=" * 70)
    print(f"TRAINING: {name.upper()}")
    print("=" * 70)

    torch.manual_seed(SEED)  # re-seed so each model is trained independently and reproducibly

    train_ds = MutationDataset(splits[name]['train'])
    val_ds = MutationDataset(splits[name]['val'])

    class_weights = compute_class_weights(train_ds.y.numpy(), device=device)
    print(f"Class weights ({CLASSES}): {class_weights.cpu().numpy().round(3)}")

    model = SimpleCNN()
    model, history = train_model(
        model, train_ds, val_ds, device,
        max_epochs=50, batch_size=128, lr=1e-4,
        patience=15, lr_patience=5, lr_factor=0.5,
        grad_clip_norm=1.0, class_weights=class_weights,
        seed=SEED, verbose=True,
    )

    checkpoint_path = os.path.join(RESULTS_DIR, name, 'checkpoint.pt')
    torch.save({
        'model_state_dict': model.state_dict(),
        'seed': SEED,
        'dataset': name,
        'window_size': WINDOW_SIZE,
        'best_val_accuracy': history['best_val_accuracy'],
        'best_epoch': history['best_epoch'],
        'stopped_epoch': history['stopped_epoch'],
        'classes': CLASSES,
    }, checkpoint_path)

    print(f"\nBest val_accuracy: {history['best_val_accuracy']:.4f} "
          f"(epoch {history['best_epoch']}, stopped at {history['stopped_epoch']})")
    print(f"Checkpoint saved -> {checkpoint_path}\n")

    models[name] = model
    histories[name] = history


TRAINING: HOTSPOT


Class weights (['C>A', 'C>G', 'C>T', 'T>A', 'T>C', 'T>G']): [0.885 1.958 0.347 2.811 1.147 3.97 ]


epoch   1/50  train_loss=1.2232  val_loss=1.7420  val_acc=0.0967  lr=1.00e-04 *


epoch   2/50  train_loss=0.9664  val_loss=1.7552  val_acc=0.1235  lr=1.00e-04 *


epoch   3/50  train_loss=0.8934  val_loss=1.8015  val_acc=0.0990  lr=1.00e-04


...

epoch  41/50  train_loss=1.1555  val_loss=1.7483  val_acc=0.3482  lr=3.13e-06
epoch  42/50  train_loss=1.1521  val_loss=1.7464  val_acc=0.3653  lr=3.13e-06


epoch  43/50  train_loss=1.1542  val_loss=1.7514  val_acc=0.3549  lr=3.13e-06
epoch  44/50  train_loss=1.1596  val_loss=1.7474  val_acc=0.3452  lr=3.13e-06


epoch  45/50  train_loss=1.1561  val_loss=1.7489  val_acc=0.3579  lr=1.56e-06
epoch  46/50  train_loss=1.1492  val_loss=1.7472  val_acc=0.3653  lr=1.56e-06


epoch  47/50  train_loss=1.1439  val_loss=1.7457  val_acc=0.3452  lr=1.56e-06
epoch  48/50  train_loss=1.1557  val_loss=1.7482  val_acc=0.3452  lr=1.56e-06


epoch  49/50  train_loss=1.1505  val_loss=1.7472  val_acc=0.3422  lr=1.56e-06
epoch  50/50  train_loss=1.1564  val_loss=1.7482  val_acc=0.3661  lr=1.56e-06

Best val_accuracy: 0.3721 (epoch 38, stopped at 50)
Checkpoint saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main\rare\checkpoint.pt



## Baselines

Random (uniform over 6 classes), majority class (from that dataset's own
training split), CpG rule (predict C>T if the 21bp window's centre/next base
form a CpG dinucleotide, else majority class), and a trinucleotide (k=3)
frequency logistic regression -- all fit on the training split and evaluated
on the test split, per dataset.

The CpG rule is checked against the majority baseline explicitly rather than
assumed to differ: since C>T is already the dominant class overall, the
CpG-context predictions and the "always majority" predictions may turn out
to be identical.


In [6]:
baseline_preds = {}
baseline_info = {}

for name in DATASETS:
    train_df = splits[name]['train']
    test_df = splits[name]['test']
    y_train = encode_labels(train_df['MutationType'].values)
    y_test = encode_labels(test_df['MutationType'].values)

    rand_preds = random_baseline(len(y_test), seed=SEED)
    maj_preds, majority_class = majority_baseline(y_train, len(y_test))
    cpg_preds, is_cpg = cpg_baseline(test_df['Sequence'].values, CLASS_TO_IDX[majority_class])
    cpg_equals_majority = bool(np.array_equal(cpg_preds, maj_preds))
    logreg_preds = logistic_regression_baseline(
        train_df['Sequence'].values, y_train, test_df['Sequence'].values, seed=SEED
    )

    baseline_preds[name] = {
        'random': rand_preds, 'majority': maj_preds,
        'cpg': cpg_preds, 'logistic': logreg_preds,
    }
    baseline_info[name] = {
        'majority_class': majority_class,
        'n_cpg_context_positions_in_test': int(is_cpg.sum()),
        'cpg_predictions_equal_majority_predictions': cpg_equals_majority,
    }

    print(f"{name}: majority class = {majority_class}, "
          f"CpG-context test rows = {int(is_cpg.sum())}/{len(test_df)}, "
          f"CpG rule == majority baseline: {cpg_equals_majority}")


hotspot: majority class = C>T, CpG-context test rows = 71555/112511, CpG rule == majority baseline: True
rare: majority class = C>T, CpG-context test rows = 104/2362, CpG rule == majority baseline: True


## Theoretical position ceiling

For each test-set `position_id`, the majority-subtype frequency among that
position's *test* instances (not the global position table's majority,
which is aggregated over train+val+test) -- averaged across positions,
weighted by instance count. Computed directly from ground truth, never from
model predictions.


In [7]:
position_ceilings = {
    name: position_majority_ceiling(splits[name]['test'])
    for name in DATASETS
}
for name, ceiling in position_ceilings.items():
    print(f"{name}: position-majority ceiling = {ceiling:.4f}")


hotspot: position-majority ceiling = 0.8032
rare: position-majority ceiling = 0.7269


## Evaluate CNN on the test set, compute all metrics, run McNemar's test


In [8]:
all_metrics = {}
all_predictions_frames = []

for name in DATASETS:
    test_df = splits[name]['test']
    test_ds = MutationDataset(test_df)
    y_test = test_ds.y.numpy()

    cnn_preds, cnn_probs = predict(models[name], test_ds, device)

    # Cluster (locus) ids for this test set, aligned 1:1 with y_test/cnn_preds,
    # via the position_id -> cluster_id map built in the split-QC cell above.
    # Needed for the cluster-resampled CI and significance tests below: many
    # instances share a locus (identical/near-identical sequence context), so
    # instance-level resampling/McNemar understates uncertainty and overstates
    # significance (see src/metrics.py docstrings).
    cluster_ids_test = position_to_cluster.loc[test_df['position_id']].values

    metrics = compute_all_metrics(y_test, cnn_preds, cluster_ids=cluster_ids_test)
    metrics['baselines'] = {
        'random_accuracy': float((baseline_preds[name]['random'] == y_test).mean()),
        'majority_accuracy': float((baseline_preds[name]['majority'] == y_test).mean()),
        'cpg_accuracy': float((baseline_preds[name]['cpg'] == y_test).mean()),
        'logistic_accuracy': float((baseline_preds[name]['logistic'] == y_test).mean()),
        'majority_class': baseline_info[name]['majority_class'],
        'n_cpg_context_positions_in_test': baseline_info[name]['n_cpg_context_positions_in_test'],
        'cpg_predictions_equal_majority_predictions': baseline_info[name]['cpg_predictions_equal_majority_predictions'],
    }
    metrics['position_majority_ceiling'] = position_ceilings[name]
    # Instance-level McNemar, kept for continuity/comparison with the original
    # analysis -- pseudoreplicated (see cluster_test below for the corrected
    # version); do not treat these p-values as valid under locus clustering.
    metrics['mcnemar'] = {
        'cnn_vs_majority': mcnemar_test(y_test, cnn_preds, baseline_preds[name]['majority']),
        'cnn_vs_logistic': mcnemar_test(y_test, cnn_preds, baseline_preds[name]['logistic']),
    }
    # Cluster-resampled paired bootstrap + Wilcoxon signed-rank, both
    # respecting the locus as the independence unit -- the corrected
    # replacement for metrics['mcnemar'] above.
    metrics['cluster_test'] = {
        'cnn_vs_majority': {
            'bootstrap': cluster_paired_bootstrap_test(
                y_test, cnn_preds, baseline_preds[name]['majority'], cluster_ids_test
            ),
            'wilcoxon': cluster_wilcoxon_paired_test(
                y_test, cnn_preds, baseline_preds[name]['majority'], cluster_ids_test
            ),
        },
        'cnn_vs_logistic': {
            'bootstrap': cluster_paired_bootstrap_test(
                y_test, cnn_preds, baseline_preds[name]['logistic'], cluster_ids_test
            ),
            'wilcoxon': cluster_wilcoxon_paired_test(
                y_test, cnn_preds, baseline_preds[name]['logistic'], cluster_ids_test
            ),
        },
    }
    metrics['n_test_instances'] = int(len(y_test))
    metrics['n_test_positions'] = int(test_df['position_id'].nunique())
    metrics['window_size'] = WINDOW_SIZE
    metrics['best_epoch'] = histories[name]['best_epoch']
    metrics['stopped_epoch'] = histories[name]['stopped_epoch']

    metrics_path = os.path.join(RESULTS_DIR, name, 'metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"{name}: metrics written -> {metrics_path}")

    cm_path = os.path.join(RESULTS_DIR, name, 'confusion_matrix.png')
    plot_confusion_matrix(metrics['confusion_matrix'], CLASSES,
                           title=f"{name} (window={WINDOW_SIZE}bp) confusion matrix", out_path=cm_path)
    print(f"{name}: confusion matrix plot -> {cm_path}")

    pred_df = pd.DataFrame({
        'position_id': test_df['position_id'].values,
        'window_size': WINDOW_SIZE,
        'dataset': name,
        'true_label': [CLASSES[i] for i in y_test],
        'predicted_label': [CLASSES[i] for i in cnn_preds],
        'probabilities': list(cnn_probs),
    })[['position_id', 'window_size', 'dataset', 'true_label', 'predicted_label', 'probabilities']]

    pred_path = os.path.join(RESULTS_DIR, name, 'predictions.parquet')
    pred_df.to_parquet(pred_path, engine='pyarrow', index=False)
    print(f"{name}: predictions -> {pred_path} ({len(pred_df):,} rows)\n")

    all_metrics[name] = metrics
    all_predictions_frames.append(pred_df)


hotspot: metrics written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main\hotspot\metrics.json
hotspot: confusion matrix plot -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main\hotspot\confusion_matrix.png


hotspot: predictions -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main\hotspot\predictions.parquet (112,511 rows)



rare: metrics written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main\rare\metrics.json
rare: confusion matrix plot -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main\rare\confusion_matrix.png
rare: predictions -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main\rare\predictions.parquet (2,362 rows)



## Assemble `results/main/summary.csv` (one row per dataset)

In [9]:
summary_rows = []
for name in DATASETS:
    m = all_metrics[name]
    summary_rows.append({
        'dataset': name,
        'accuracy': m['accuracy'],
        'accuracy_ci_low': m['accuracy_ci_low'],
        'accuracy_ci_high': m['accuracy_ci_high'],
        'accuracy_ci_low_clustered': m['accuracy_ci_low_clustered'],
        'accuracy_ci_high_clustered': m['accuracy_ci_high_clustered'],
        'balanced_accuracy': m['balanced_accuracy'],
        'macro_f1': m['macro_f1'],
        'weighted_f1': m['weighted_f1'],
        'mcc': m['mcc'],
        'majority_accuracy': m['baselines']['majority_accuracy'],
        'random_accuracy': m['baselines']['random_accuracy'],
        'cpg_accuracy': m['baselines']['cpg_accuracy'],
        'logistic_accuracy': m['baselines']['logistic_accuracy'],
        'position_majority_ceiling': m['position_majority_ceiling'],
        'mcnemar_cnn_vs_majority_pvalue': m['mcnemar']['cnn_vs_majority']['pvalue'],
        'cluster_bootstrap_cnn_vs_majority_pvalue': m['cluster_test']['cnn_vs_majority']['bootstrap']['pvalue'],
        'cluster_wilcoxon_cnn_vs_majority_pvalue': m['cluster_test']['cnn_vs_majority']['wilcoxon']['pvalue'],
    })

summary_df = pd.DataFrame(summary_rows, columns=[
    'dataset', 'accuracy', 'accuracy_ci_low', 'accuracy_ci_high',
    'accuracy_ci_low_clustered', 'accuracy_ci_high_clustered', 'balanced_accuracy',
    'macro_f1', 'weighted_f1', 'mcc', 'majority_accuracy', 'random_accuracy',
    'cpg_accuracy', 'logistic_accuracy', 'position_majority_ceiling',
    'mcnemar_cnn_vs_majority_pvalue', 'cluster_bootstrap_cnn_vs_majority_pvalue',
    'cluster_wilcoxon_cnn_vs_majority_pvalue',
])

summary_path = os.path.join(RESULTS_DIR, 'summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f"Summary written -> {summary_path}\n")
summary_df


Summary written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main\summary.csv



,dataset,accuracy,accuracy_ci_low,accuracy_ci_high,accuracy_ci_low_clustered,accuracy_ci_high_clustered,balanced_accuracy,macro_f1,weighted_f1,mcc,majority_accuracy,random_accuracy,cpg_accuracy,logistic_accuracy,position_majority_ceiling,mcnemar_cnn_vs_majority_pvalue,cluster_bootstrap_cnn_vs_majority_pvalue,cluster_wilcoxon_cnn_vs_majority_pvalue
0,hotspot,0.132200,0.130272,0.134130,0.053702,0.238118,0.259782,0.155013,0.093986,0.066582,0.649385,0.165744,0.649385,0.193403,0.803228,0.000000e+00,0.014,0.007379
1,rare,0.218036,0.201937,0.235817,0.151688,0.292324,0.198719,0.194176,0.221522,0.026101,0.358171,0.161727,0.358171,0.354361,0.726926,2.369098e-28,0.028,0.087506


## Sanity check against previously published numbers

These numbers are from a prior version of this experiment, run on the
dataset *before* the position-ID / shared-N-filter bug fix in notebook 00.
They won't match exactly -- the underlying dataset has changed -- but the
rough magnitude and ordering (majority baseline direction, MCC sign, which
dataset is harder) should be similar. A wildly different result (e.g. a
flipped majority-baseline direction, or a strongly positive MCC where the
prior run was near-zero/negative) would point to a pipeline bug, not a
genuine result change, and is flagged explicitly below rather than silently
reported.


In [10]:
published = {
    'hotspot': {'accuracy': 0.2382, 'mcc': 0.009, 'balanced_accuracy': 0.1452,
                'macro_f1': 0.132, 'majority_accuracy': 0.6315},
    'rare': {'accuracy': 0.1312, 'mcc': -0.053, 'balanced_accuracy': 0.1303,
             'macro_f1': 0.126, 'majority_accuracy': 0.3224},
}

FLAG_THRESHOLD = 0.15  # absolute difference beyond which a metric is flagged for review

print(f"{'dataset':<8} {'metric':<20} {'published':>10} {'current':>10} {'abs diff':>10}  flag")
print("-" * 70)
flags = []
for name in DATASETS:
    cur = all_metrics[name]
    comparisons = {
        'accuracy': cur['accuracy'],
        'mcc': cur['mcc'],
        'balanced_accuracy': cur['balanced_accuracy'],
        'macro_f1': cur['macro_f1'],
        'majority_accuracy': cur['baselines']['majority_accuracy'],
    }
    for metric_name, cur_val in comparisons.items():
        pub_val = published[name][metric_name]
        diff = abs(cur_val - pub_val)
        sign_flip = (pub_val > 0) != (cur_val > 0) and abs(pub_val) > 1e-6 and abs(cur_val) > 1e-6
        flagged = diff > FLAG_THRESHOLD or sign_flip
        marker = '<-- FLAGGED' if flagged else ''
        print(f"{name:<8} {metric_name:<20} {pub_val:>10.4f} {cur_val:>10.4f} {diff:>10.4f}  {marker}")
        if flagged:
            flags.append((name, metric_name, pub_val, cur_val))

print()
if flags:
    print(f"FLAGGED: {len(flags)} metric(s) differ substantially from the published run "
          "-- investigate before trusting this pipeline's results:")
    for name, metric_name, pub_val, cur_val in flags:
        print(f"  {name}/{metric_name}: published={pub_val:.4f}, current={cur_val:.4f}")
else:
    print("No metric deviates enough from the published run to flag -- "
          "same rough ballpark and ordering as expected.")


dataset  metric                published    current   abs diff  flag
----------------------------------------------------------------------
hotspot  accuracy                 0.2382     0.1322     0.1060  
hotspot  mcc                      0.0090     0.0666     0.0576  
hotspot  balanced_accuracy        0.1452     0.2598     0.1146  
hotspot  macro_f1                 0.1320     0.1550     0.0230  
hotspot  majority_accuracy        0.6315     0.6494     0.0179  
rare     accuracy                 0.1312     0.2180     0.0868  
rare     mcc                     -0.0530     0.0261     0.0791  <-- FLAGGED
rare     balanced_accuracy        0.1303     0.1987     0.0684  
rare     macro_f1                 0.1260     0.1942     0.0682  
rare     majority_accuracy        0.3224     0.3582     0.0358  

FLAGGED: 1 metric(s) differ substantially from the published run -- investigate before trusting this pipeline's results:
  rare/mcc: published=-0.0530, current=0.0261


## Final summary table

In [11]:
print("=" * 100)
print("FINAL RESULTS -- 01_main_experiment (window=21bp)")
print("=" * 100)
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)
print(summary_df.to_string(index=False))

print("\n" + "=" * 100)
print("metrics.json contents")
print("=" * 100)
for name in DATASETS:
    print(f"\n--- {name} ---")
    print(json.dumps(all_metrics[name], indent=2))


FINAL RESULTS -- 01_main_experiment (window=21bp)
dataset  accuracy  accuracy_ci_low  accuracy_ci_high  accuracy_ci_low_clustered  accuracy_ci_high_clustered  balanced_accuracy  macro_f1  weighted_f1      mcc  majority_accuracy  random_accuracy  cpg_accuracy  logistic_accuracy  position_majority_ceiling  mcnemar_cnn_vs_majority_pvalue  cluster_bootstrap_cnn_vs_majority_pvalue  cluster_wilcoxon_cnn_vs_majority_pvalue
hotspot  0.132200         0.130272          0.134130                   0.053702                    0.238118           0.259782  0.155013     0.093986 0.066582           0.649385         0.165744      0.649385           0.193403                   0.803228                    0.000000e+00                                     0.014                                 0.007379
   rare  0.218036         0.201937          0.235817                   0.151688                    0.292324           0.198719  0.194176     0.221522 0.026101           0.358171         0.161727      0.358171  